# Passos Mágicos — Monitoramento de Drift do Modelo

Este notebook analisa o comportamento do modelo em produção comparando:
- Distribuição das features de entrada vs. distribuição de treino
- Distribuição das predições ao longo do tempo

**Drift detectado** quando o teste KS retorna `p-value < 0.05` para uma feature.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from src.preprocessing import load_raw_data, clean_data, NUMERIC_FEATURES
from src.feature_engineering import create_features

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

ALPHA = 0.05  # nível de significância para detecção de drift
print('Setup OK')

---
## 1. Distribuição de Referência (Treino)

Carrega os dados de treino para servir como linha de base.

In [ ]:
df_raw = load_raw_data()
df_ref = clean_data(df_raw)
df_ref = create_features(df_ref)

feature_cols = [c for c in NUMERIC_FEATURES if c in df_ref.columns]
X_ref = df_ref[feature_cols].dropna()

print(f'Distribuição de referência: {X_ref.shape[0]} amostras, {X_ref.shape[1]} features')
X_ref.describe().round(2)

---
## 2. Dados de Produção (Predições Registradas)

Em produção, cada chamada ao `/predict` gera um log com as features de entrada.
Os logs podem ser exportados do **Amazon CloudWatch** ou de um arquivo local.

Para esta demonstração, simulamos um lote de predições recentes com drift intencional em algumas features.

In [ ]:
def simulate_production_data(n: int = 200, drift_features: list = None, drift_shift: float = 1.5) -> pd.DataFrame:
    """Simula dados de produção baseados na distribuição de treino.
    Se drift_features for fornecido, aplica um shift nas médias dessas features.
    """
    rng = np.random.default_rng(seed=99)
    rows = []
    for _ in range(n):
        idx = rng.integers(0, len(X_ref))
        row = X_ref.iloc[idx].copy()
        # Adiciona ruído para simular variação natural
        noise = rng.normal(0, 0.3, size=len(row))
        row = row + noise
        rows.append(row)

    df_prod = pd.DataFrame(rows, columns=X_ref.columns)

    if drift_features:
        for feat in drift_features:
            if feat in df_prod.columns:
                df_prod[feat] = df_prod[feat] + drift_shift * X_ref[feat].std()

    # Simula timestamps
    df_prod['timestamp'] = pd.date_range('2025-01-01', periods=n, freq='1h')
    return df_prod


# Simula drift nas features 'inde' e 'iaa'
df_prod = simulate_production_data(n=200, drift_features=['inde', 'iaa'], drift_shift=1.5)

print(f'Dados de produção simulados: {df_prod.shape[0]} amostras')
df_prod[feature_cols].describe().round(2)

---
## 3. Teste de Kolmogorov-Smirnov por Feature

Compara a distribuição de cada feature entre treino (referência) e produção.

In [ ]:
drift_results = []

for feat in feature_cols:
    ref_vals  = X_ref[feat].dropna().values
    prod_vals = df_prod[feat].dropna().values
    ks_stat, p_value = stats.ks_2samp(ref_vals, prod_vals)
    drift_detected = p_value < ALPHA
    drift_results.append({
        'feature': feat,
        'ks_statistic': round(ks_stat, 4),
        'p_value': round(p_value, 4),
        'drift_detectado': drift_detected,
        'ref_mean': round(ref_vals.mean(), 3),
        'prod_mean': round(prod_vals.mean(), 3),
        'shift_abs': round(abs(prod_vals.mean() - ref_vals.mean()), 3),
    })

df_drift = pd.DataFrame(drift_results).sort_values('p_value')

n_drift = df_drift['drift_detectado'].sum()
print(f'\nFeatures com drift detectado (p < {ALPHA}): {n_drift} / {len(df_drift)}')
print()

def highlight_drift(row):
    color = 'background-color: #ffcccc' if row['drift_detectado'] else ''
    return [color] * len(row)

df_drift.style.apply(highlight_drift, axis=1).format({
    'ks_statistic': '{:.4f}',
    'p_value': '{:.4f}',
    'ref_mean': '{:.3f}',
    'prod_mean': '{:.3f}',
    'shift_abs': '{:.3f}',
})

---
## 4. Painel Visual de Drift por Feature

In [ ]:
ncols = 4
nrows = (len(feature_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    ax = axes[i]
    row = df_drift[df_drift['feature'] == feat].iloc[0]
    has_drift = row['drift_detectado']

    ax.hist(X_ref[feat].dropna(), bins=25, alpha=0.6, color='#4C9BE8', label='Treino (ref)', density=True)
    ax.hist(df_prod[feat].dropna(), bins=25, alpha=0.6, color='#E8774C', label='Produção', density=True)

    title_color = '#cc0000' if has_drift else '#2a2a2a'
    drift_label = ' ⚠ DRIFT' if has_drift else ''
    ax.set_title(f'{feat}{drift_label}\np={row["p_value"]:.4f}', fontsize=9, color=title_color)
    ax.legend(fontsize=7)

    if has_drift:
        for spine in ax.spines.values():
            spine.set_edgecolor('#cc0000')
            spine.set_linewidth(2)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f'Distribuição: Treino vs. Produção — {n_drift} feature(s) com drift detectado', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Sumário de Drift — Estatística KS

In [ ]:
df_plot = df_drift.sort_values('ks_statistic', ascending=True)
colors = ['#cc0000' if d else '#4C9BE8' for d in df_plot['drift_detectado']]

fig, ax = plt.subplots(figsize=(9, len(df_plot) * 0.45 + 1))
bars = ax.barh(df_plot['feature'], df_plot['ks_statistic'], color=colors, edgecolor='white')
ax.axvline(0.1, color='gray', linestyle='--', linewidth=1, label='Limiar KS = 0.1')

for bar, p in zip(bars, df_plot['p_value']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'p={p:.3f}', va='center', fontsize=8)

ax.set_xlabel('Estatística KS')
ax.set_title('Drift por Feature (vermelho = drift detectado, p < 0.05)', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 6. Evolução das Predições ao Longo do Tempo

Monitora se a proporção de predições de alto risco está mudando.

In [ ]:
# Simula predições ao longo do tempo
rng = np.random.default_rng(seed=42)
n_preds = 200

timestamps = pd.date_range('2025-01-01', periods=n_preds, freq='1h')
# Simula drift gradual: proporção de alto risco aumenta ao longo do tempo
base_prob = np.linspace(0.3, 0.7, n_preds)
probs = np.clip(base_prob + rng.normal(0, 0.1, n_preds), 0, 1)

df_preds = pd.DataFrame({'timestamp': timestamps, 'risco_defasagem': probs})
df_preds['classificacao'] = pd.cut(
    df_preds['risco_defasagem'],
    bins=[0, 0.35, 0.65, 1.0],
    labels=['Baixo Risco', 'Medio Risco', 'Alto Risco'],
    include_lowest=True
)
df_preds['semana'] = df_preds['timestamp'].dt.to_period('W')

# Taxa de alto risco por semana
weekly = df_preds.groupby('semana').apply(
    lambda x: (x['classificacao'] == 'Alto Risco').mean()
).reset_index()
weekly.columns = ['semana', 'taxa_alto_risco']
weekly['semana_str'] = weekly['semana'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Evolução da probabilidade média
df_preds.set_index('timestamp')['risco_defasagem'].rolling('24h').mean().plot(
    ax=axes[0], color='#4C9BE8', linewidth=1.5
)
axes[0].axhline(0.35, color='orange', linestyle='--', linewidth=1, label='Limiar médio risco')
axes[0].axhline(0.65, color='red', linestyle='--', linewidth=1, label='Limiar alto risco')
axes[0].set_title('Probabilidade média de risco (rolling 24h)', fontsize=11)
axes[0].set_ylabel('P(risco)')
axes[0].legend(fontsize=8)

# Taxa de alto risco por semana
axes[1].bar(weekly['semana_str'], weekly['taxa_alto_risco'] * 100,
            color='#E8774C', edgecolor='white')
axes[1].set_title('Taxa de "Alto Risco" por semana (%)', fontsize=11)
axes[1].set_ylabel('% predições Alto Risco')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Monitoramento Temporal das Predições', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Resumo do Monitoramento

In [ ]:
print('=' * 55)
print('  RESUMO DO MONITORAMENTO DE DRIFT')
print('=' * 55)
print(f'  Amostras de referência (treino) : {len(X_ref)}')
print(f'  Amostras de produção analisadas : {len(df_prod)}')
print(f'  Nível de significância (alpha)  : {ALPHA}')
print(f'  Features monitoradas            : {len(feature_cols)}')
print(f'  Features com drift detectado    : {n_drift}')
print('=' * 55)

if n_drift > 0:
    print('\n  Features com drift (p < 0.05):')
    for _, row in df_drift[df_drift['drift_detectado']].iterrows():
        print(f'    - {row["feature"]:<25} KS={row["ks_statistic"]:.4f}  p={row["p_value"]:.4f}')
    print('\n  AÇÃO RECOMENDADA: Investigar origem do drift e considerar retreinamento.')
else:
    print('\n  Nenhum drift detectado. Modelo estável.')

print('=' * 55)